In [0]:
#Gold Exercise Recommendations
from pyspark.sql.functions import col, lower, trim, when, lit, concat_ws

gym = spark.table("main.default.silver_gym_exercises")

gold_exercises = (
    gym
    .withColumn("exercise_name", trim(col("exercise_name")))
    .withColumn("body_part", lower(trim(col("body_part"))))
    .withColumn("equipment", lower(trim(col("equipment"))))
    .withColumn("difficulty_level", lower(trim(col("difficulty_level"))))
    .withColumn(
        "goal_tag",
        when(col("body_part").contains("legs"), "speed_agility")
        .when(col("body_part").contains("chest"), "strength")
        .when(col("body_part").contains("back"), "strength")
        .when(col("body_part").contains("shoulders"), "strength")
        .when(col("body_part").contains("abdominals"), "core")
        .otherwise("general_fitness")
    )
    .withColumn(
        "agent_text",
        concat_ws(
            " | ",
            col("exercise_name"),
            col("exercise_type"),
            col("body_part"),
            col("equipment"),
            col("difficulty_level"),
            col("description")
        )
    )
    .filter(col("exercise_name").isNotNull())
)

gold_exercises.write.mode("overwrite").saveAsTable(
    "main.default.gold_exercise_recommendations"
)

display(gold_exercises.limit(10))

exercise_name,description,exercise_type,body_part,equipment,difficulty_level,goal_tag,agent_text
FYR2 Kettlebell Juggle,No description available,Strength,abdominals,kettlebells,intermediate,core,FYR2 Kettlebell Juggle | Strength | abdominals | kettlebells | intermediate | No description available
Holman Boat with Feet Push-Out and Overhead Press,No description available,Strength,abdominals,dumbbell,intermediate,core,Holman Boat with Feet Push-Out and Overhead Press | Strength | abdominals | dumbbell | intermediate | No description available
Holman Weighted Burpee to Side Delt Raise,No description available,Strength,abdominals,dumbbell,intermediate,core,Holman Weighted Burpee to Side Delt Raise | Strength | abdominals | dumbbell | intermediate | No description available
Landmine twist,"The landmine twist is a rotational abdominal movement performed using an angled barbell anchored at floor level in a landmine device. It can also be performed by sticking a barbell in the corner of a room, preferably in a towel to protect the walls. It targets the deep muscles of the core, including both the obliques and the transversus abdominis. It can be done fast or slow, for time or reps, either in traditional muscle-focused rep ranges such as 8-12 reps per side or for higher rep ranges.",Strength,abdominals,other,intermediate,core,"Landmine twist | Strength | abdominals | other | intermediate | The landmine twist is a rotational abdominal movement performed using an angled barbell anchored at floor level in a landmine device. It can also be performed by sticking a barbell in the corner of a room, preferably in a towel to protect the walls. It targets the deep muscles of the core, including both the obliques and the transversus abdominis. It can be done fast or slow, for time or reps, either in traditional muscle-focused rep ranges such as 8-12 reps per side or for higher rep ranges."
Pallof press,"""The Pallof press is an isometric exercise that trains core stability. It involves resisting rotation from a cable or band, developing what is sometimes called """"anti-rotation"""" core strength. It is most often seen in programs for athletes who compete in sports that test strength",power,"and functional movements. it can be trained in timed holds or for reps by pressing the cable or band away from the body.""",strength,abdominals,general_fitness,"Pallof press | power | and functional movements. it can be trained in timed holds or for reps by pressing the cable or band away from the body."" | strength | abdominals | ""The Pallof press is an isometric exercise that trains core stability. It involves resisting rotation from a cable or band, developing what is sometimes called """"anti-rotation"""" core strength. It is most often seen in programs for athletes who compete in sports that test strength"
Ab Twist - Gethin Variation,"""The cross-body sit-up is a bodyweight exercise targeting the ab muscles, the obliques and rectus abdominis or """"six-pack"""" muscles in particular. It involves bringing the opposite-side knee and elbow together","usually alternating sides with each rep. It can be performed for time or reps as part of the ab-focused portion of any workout.""",strength,abdominals,other,general_fitness,"Ab Twist - Gethin Variation | usually alternating sides with each rep. It can be performed for time or reps as part of the ab-focused portion of any workout."" | strength | abdominals | other | ""The cross-body sit-up is a bodyweight exercise targeting the ab muscles, the obliques and rectus abdominis or """"six-pack"""" muscles in particular. It involves bringing the opposite-side knee and elbow together"
Ab Wheel Roll-Out - Gethin Variation,"The ab wheel roll-out is an exercise targeting the abdominals, often with an inexpensive wheel device with a handle on either side. Many trainers and strength coaches claim it's one of, if no the best exercise for developing strength in the midsection. Beginners may have to start with a limited range

In [0]:
#Gold Basketball Benchmarks

from pyspark.sql.functions import avg, col

nba = spark.table("main.default.silver_nba_stats")

gold_basketball_benchmarks = (
    nba
    .filter(col("season_year") >= 2000)
    .filter(col("position").isNotNull())
    .groupBy("position")
    .agg(
        avg("points").alias("avg_points"),
        avg("assists").alias("avg_assists"),
        avg("rebounds").alias("avg_rebounds"),
        avg("steals").alias("avg_steals"),
        avg("blocks").alias("avg_blocks"),
        avg("minutes_played").alias("avg_minutes_played")
    )
)

gold_basketball_benchmarks.write.mode("overwrite").saveAsTable(
    "main.default.gold_basketball_benchmarks"
)

display(gold_basketball_benchmarks)

position,avg_points,avg_assists,avg_rebounds,avg_steals,avg_blocks,avg_minutes_played
PG,457.8745779064158,197.8880849011095,116.98408104196817,43.38543174143753,6.862518089725036,1137.9734684032803
PF,447.72740384615383,63.96730769230769,259.16923076923075,30.52355769230769,29.283173076923077,1092.8033653846153
SG,515.2147922998987,107.87487335359675,138.97517730496455,40.31712259371834,11.593718338399189,1194.7330293819655
SF,493.2193958664547,85.6290408055114,190.39056703762586,39.05564387917329,19.021727609962905,1208.0068892421834
C,365.2830466830467,45.64520884520884,267.5223587223587,23.19213759213759,45.63046683046683,981.5007371007371
SG-PG,526.1111111111111,183.77777777777777,116.38888888888889,40.333333333333336,6.277777777777778,1263.2222222222222
C-PF,293.7826086956522,35.608695652173914,229.17391304347825,26.652173913043477,36.34782608695652,883.304347826087
SF-SG,647.8888888888889,119.55555555555556,197.72222222222223,54.666666666666664,19.22222222222222,1541.111111111111
PG-SG,497.08,169.88,120.84,41.52,6.24,1209.48
SG-SF,396.94444444444446,82.44444444444444,130.61111111111111,32.5,9.11111111111111,1114.0


In [0]:
gold_basketball_benchmarks.printSchema()

root
 |-- position: string (nullable = true)
 |-- avg_points: double (nullable = true)
 |-- avg_assists: double (nullable = true)
 |-- avg_rebounds: double (nullable = true)
 |-- avg_steals: double (nullable = true)
 |-- avg_blocks: double (nullable = true)
 |-- avg_minutes_played: double (nullable = true)



In [0]:
#Gold Nutrition Lookup 

food = spark.table("main.default.silver_food_nutrients")

gold_nutrition_lookup = (
    food
    .dropDuplicates()
)

gold_nutrition_lookup.write.mode("overwrite").saveAsTable(
    "main.default.gold_nutrition_lookup"
)

display(gold_nutrition_lookup.limit(10))

food_nutrient_id,adjusted_amount,lab_method_id,nutrient_name
2201861,0.37,1015,Pantothenic Acid
2201867,81.4,1009,Magnesium
2201887,57.6,1007,Moisture
2201890,38.0,1009,Calcium
2201897,null,1042,14:1 Tetradecenoic (Myristoleic)
2201905,null,1042,22:4 Docosatetraenoic
2201935,null,1042,"20:3 8,11,14-Eicosatrienoic (gamma)"
2201942,18.2,1053,Fat
2201975,0.046,1042,20:1 Eicosenoic (incl. Gadoleic)
2201980,0.004,1042,20:2 Eicosadienoic


In [0]:
#Gold Saftey Rules -- 

from pyspark.sql import Row

safety_rules = [
    Row(rule_id="R001", rule_category="age", rule_text="Athletes ages 13-17 require parent or coach oversight."),
    Row(rule_id="R002", rule_category="injury", rule_text="If injury status is not none, avoid high-impact training and recommend coach or medical review."),
    Row(rule_id="R003", rule_category="workload", rule_text="Youth athletes should not receive intense training plans for more than 5 days per week."),
    Row(rule_id="R004", rule_category="nutrition", rule_text="Do not recommend supplements, extreme diets, fasting, or medical nutrition treatment."),
    Row(rule_id="R005", rule_category="recovery", rule_text="Every weekly plan must include recovery, stretching, hydration, and sleep guidance."),
    Row(rule_id="R006", rule_category="safety", rule_text="The agent provides educational guidance only, not medical advice.")
]

gold_safety_rules = spark.createDataFrame(safety_rules)

gold_safety_rules.write.mode("overwrite").saveAsTable(
    "main.default.gold_safety_rules"
)

display(gold_safety_rules)

rule_id,rule_category,rule_text
R001,age,Athletes ages 13-17 require parent or coach oversight.
R002,injury,"If injury status is not none, avoid high-impact training and recommend coach or medical review."
R003,workload,Youth athletes should not receive intense training plans for more than 5 days per week.
R004,nutrition,"Do not recommend supplements, extreme diets, fasting, or medical nutrition treatment."
R005,recovery,"Every weekly plan must include recovery, stretching, hydration, and sleep guidance."
R006,safety,"The agent provides educational guidance only, not medical advice."


In [0]:
# Gold food portions

food_portions = spark.table("main.default.silver_food_portions")

gold_food_portions = (
    food_portions
    .dropDuplicates()
)

gold_food_portions.write.mode("overwrite").saveAsTable(
    "main.default.gold_food_portions"
)

display(gold_food_portions.limit(10))

id,fdc_id,seq_num,amount,measure_unit_id,portion_description,modifier,gram_weight,data_points,footnote,min_year_acquired
118703,319880,null,2.0,1001,null,null,36.2,1,null,null
118717,319979,null,2.0,1001,null,null,30.4,1,null,null
118748,320314,null,1.0,1000,null,null,232.0,1,null,null
118758,320361,null,1.0,1046,null,raw,707.0,2,null,null
118793,320481,null,1.0,1000,null,null,147.0,1,null,null
118817,321401,null,1.0,1002,null,null,6.2,1,null,null
118818,321407,null,1.0,1002,null,null,6.0,1,null,null
118827,321459,null,1.0,1002,null,null,6.0,1,null,null
118828,321465,null,1.0,1002,null,null,5.9,1,null,null
118832,321494,null,1.0,1002,null,null,6.0,1,null,null


In [0]:
# Now Let's Verify the Gold Agent Ready Tables -- 

spark.sql("SHOW TABLES IN main.default").show(truncate=False)

+--------+-----------------------------+-----------+
|database|tableName                    |isTemporary|
+--------+-----------------------------+-----------+
|default |assignment_file              |false      |
|default |bronze_athlete_profiles      |false      |
|default |bronze_food_categories       |false      |
|default |bronze_food_nutrients        |false      |
|default |bronze_food_portions         |false      |
|default |bronze_food_sample           |false      |
|default |bronze_gym_exercises         |false      |
|default |bronze_nba_stats             |false      |
|default |bronze_player_data           |false      |
|default |bronze_players               |false      |
|default |gold_basketball_benchmarks   |false      |
|default |gold_exercise_recommendations|false      |
|default |gold_food_portions           |false      |
|default |gold_nutrition_lookup        |false      |
|default |gold_safety_rules            |false      |
|default |silver_athlete_profiles      |false 

In [0]:
#Quick Agent Readiness Check -- 

gold_tables = [
    "gold_exercise_recommendations",
    "gold_basketball_benchmarks",
    "gold_nutrition_lookup",
    "gold_food_portions",
    "gold_safety_rules"
]

for table in gold_tables:
    df = spark.table(f"main.default.{table}")
    print(f"{table}: {df.count()} rows, {len(df.columns)} columns")

gold_exercise_recommendations: 2918 rows, 8 columns
gold_basketball_benchmarks: 16 rows, 7 columns
gold_nutrition_lookup: 134267 rows, 4 columns
gold_food_portions: 10951 rows, 11 columns
gold_safety_rules: 6 rows, 3 columns
